<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

// ИНТЕРФЕЙСЫ
public interface INotifiable
{
    void SendNotification(string message);
}

public interface ILoggable
{
    void Log(string message);
}

// СЕРВИСЫ (управление зависимостями)
public class NotificationService : INotifiable
{
    public void SendNotification(string message)
    {
        Console.WriteLine($"[Уведомление]: {message}");
    }
}

public class LoggerService : ILoggable
{
    public void Log(string message)
    {
        Console.WriteLine($"[Лог]: {message}");
    }
}

// БАЗОВЫЙ КЛАСС
public class Task
{
    public int TaskId { get; set; }
    public string TaskName { get; set; }
    public string Priority { get; set; }
    public bool IsCompleted { get; set; }
    public string AssignedTo { get; set; }

    // Новые атрибуты
    public string Category { get; set; }
    public int EstimatedHours { get; set; }
    public double Progress { get; set; }
    public List<string> Tags { get; set; }
    public string Description { get; set; }
    public DateTime CreatedAt { get; set; }
    public List<string> Comments { get; set; }

    // Зависимости
    protected readonly INotifiable notifier;
    protected readonly ILoggable logger;

    public Task(int taskId, string taskName, string priority, string assignedTo, INotifiable notifyService, ILoggable logService)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        AssignedTo = assignedTo;
        CreatedAt = DateTime.Now;
        Comments = new List<string>();
        Tags = new List<string>();
        notifier = notifyService;
        logger = logService;
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Progress = 100;
        Console.WriteLine($"Задача '{TaskName}' выполнена.");
        notifier.SendNotification($"Задача '{TaskName}' завершена!");
        logger.Log($"Task {TaskId} completed by {AssignedTo}");
    }

    public virtual void GetTaskDetails()
    {
        Console.WriteLine($"ID: {TaskId}, Название: {TaskName}, Приоритет: {Priority}, Исполнитель: {AssignedTo}");
        Console.WriteLine($"Статус: {(IsCompleted ? "выполнена" : "в процессе")}, Прогресс: {Progress}%");
    }

    public virtual void ReassignTo(string newAssign)
    {
        AssignedTo = newAssign;
        Console.WriteLine($"Задача '{TaskName}' переназначена исполнителю: {newAssign}");
        notifier.SendNotification($"Задача '{TaskName}' теперь назначена {newAssign}");
    }

    public void AddComment(string comment)
    {
        Comments.Add(comment);
        logger.Log($"Комментарий к задаче {TaskId}: {comment}");
    }

    public void ShowComments()
    {
        Console.WriteLine("Комментарии:");
        foreach (var c in Comments)
            Console.WriteLine($" - {c}");
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
        logger.Log($"Добавлен тег '{tag}' к задаче {TaskId}");
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 1
public class DelegateTask : Task
{
    public DateTime DueDate { get; set; }
    public int ReminderDays { get; set; }
    public bool IsUrgent { get; set; }
    public string Supervisor { get; set; }

    public DelegateTask(int taskId, string taskName, string priority, string assignedTo,
        DateTime dueDate, INotifiable notifier, ILoggable logger)
        : base(taskId, taskName, priority, assignedTo, notifier, logger)
    {
        DueDate = dueDate;
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        Console.WriteLine($"Дата завершения: {DateTime.Now.ToShortDateString()} (срок был {DueDate.ToShortDateString()})");
    }

    public void SetReminder(int daysBefore)
    {
        ReminderDays = daysBefore;
        notifier.SendNotification($"Напоминание по задаче '{TaskName}' за {daysBefore} дней до срока.");
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 2
public class TeamTask : Task
{
    public string TeamName { get; set; }
    public List<string> TeamMembers { get; set; }
    public string ProjectName { get; set; }

    public TeamTask(int taskId, string taskName, string priority, string assignedTo, string teamName,
        INotifiable notifier, ILoggable logger)
        : base(taskId, taskName, priority, assignedTo, notifier, logger)
    {
        TeamName = teamName;
        TeamMembers = new List<string>();
    }

    public override void ReassignTo(string newAssignee)
    {
        base.ReassignTo(newAssignee);
        logger.Log($"Team task reassigned to {newAssignee}");
    }

    public void AddMember(string member)
    {
        TeamMembers.Add(member);
        logger.Log($"Добавлен участник {member} в команду {TeamName}");
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 3
public interface IResearchable
{
    void AnalyzeData();
    void GenerateReport();
}

public class ResearchTask : Task, IResearchable
{
    public string DataSource { get; set; }
    public int DataVolumeMB { get; set; }

    public ResearchTask(int taskId, string taskName, string priority, string assignedTo, string dataSource,
        INotifiable notifier, ILoggable logger)
        : base(taskId, taskName, priority, assignedTo, notifier, logger)
    {
        DataSource = dataSource;
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"Источник данных: {DataSource}, Объём: {DataVolumeMB} МБ");
    }

    public void AnalyzeData()
    {
        logger.Log($"Анализ данных из {DataSource}");
    }

    public void GenerateReport()
    {
        notifier.SendNotification($"Отчёт по задаче '{TaskName}' успешно создан.");
    }
}

// УПРАВЛЕНИЕ
public class TaskManager<T> where T : Task
{
    private readonly List<T> tasks = new List<T>();

    public void AddTask(T task)
    {
        tasks.Add(task);
        Console.WriteLine($"Добавлена задача: {task.TaskName}");
    }

    public void ShowAllTasks()
    {
        Console.WriteLine("\n=== Список задач ===");
        foreach (var t in tasks)
            t.GetTaskDetails();
    }
}

// ТЕСТ
var notifier = new NotificationService();
var logger = new LoggerService();

var t1 = new DelegateTask(1, "Подготовить отчёт", "Высокий", "Дмитрий", DateTime.Now.AddDays(2), notifier, logger);
t1.SetReminder(2);
t1.MarkAsComplete();

var t2 = new TeamTask(2, "Разработка API", "Средний", "Кирилл", "Backend", notifier, logger);
t2.AddMember("Сергей");
t2.ReassignTo("Сергей");

var t3 = new ResearchTask(3, "Анализ рынка", "Низкий", "Мария", "Google Trends", notifier, logger);
t3.AnalyzeData();
t3.GenerateReport();

var manager = new TaskManager<Task>();
manager.AddTask(t1);
manager.AddTask(t2);
manager.AddTask(t3);
manager.ShowAllTasks();


[Уведомление]: Напоминание по задаче 'Подготовить отчёт' за 2 дней до срока.
Задача 'Подготовить отчёт' выполнена.
[Уведомление]: Задача 'Подготовить отчёт' завершена!
[Лог]: Task 1 completed by Дмитрий
Дата завершения: 11/6/2025 (срок был 11/8/2025)
[Лог]: Добавлен участник Сергей в команду Backend
Задача 'Разработка API' переназначена исполнителю: Сергей
[Уведомление]: Задача 'Разработка API' теперь назначена Сергей
[Лог]: Team task reassigned to Сергей
[Лог]: Анализ данных из Google Trends
[Уведомление]: Отчёт по задаче 'Анализ рынка' успешно создан.
Добавлена задача: Подготовить отчёт
Добавлена задача: Разработка API
Добавлена задача: Анализ рынка

=== Список задач ===
ID: 1, Название: Подготовить отчёт, Приоритет: Высокий, Исполнитель: Дмитрий
Статус: выполнена, Прогресс: 100%
ID: 2, Название: Разработка API, Приоритет: Средний, Исполнитель: Сергей
Статус: в процессе, Прогресс: 0%
ID: 3, Название: Анализ рынка, Приоритет: Низкий, Исполнитель: Мария
Статус: в процессе, Прогресс: 0%